In [25]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

folder=Path("/Users/yashraj/Downloads/names")
def dataIngestion(folder):
    dfs=[]
    for year in range(1880,2025):
        fname=folder/f"yob{year}.csv"
        df_year=pd.read_csv(fname,header=None,names=["Name","Gender","Count"])
        df_year['Count'] = pd.to_numeric(df_year['Count'], errors='coerce')
        df_year["Year"]=year
        dfs.append(df_year)
    df=pd.concat(dfs,ignore_index=True)
    df=df.dropna()
    df["Name"]=df["Name"].str.lower()
    agg=(df.groupby(["Name","Gender"],as_index=False)["Count"].sum())
    name_gender=agg[["Name","Gender","Count"]]
    return name_gender
    # print(name_gender.head())
    # print(name_gender.shape)
    # print(name_gender[name_gender["Name"] == "michael"])
#   Feature Engineering
def VectorisedFeature(name_gender):
    df=name_gender.copy()
    df["first_name"]=df["Name"].str[0]
    df["last_two_letters"]=df["Name"].str[-2:]
    df["length"]=df["Name"].str.len()
    df["Vowels_Count"]=df["Name"].str.count(r"[aeiou]",flags=re.I)
    return df

def split(df):
    n=len(df)
    rng=np.random.default_rng(42)
    indices=rng.permutation(n)
    n_test=(int)(n*0.2)

    test_idx=indices[:n_test]
    train_idx=indices[n_test:]
    train=df.iloc[train_idx].reset_index(drop=True)
    test=df.iloc[test_idx].reset_index(drop=True)

    return train,test

def Prior(train):
    boy_train=train.loc[train["Gender"]=="M","Count"].sum()
    girl_train=train.loc[train["Gender"]=="F","Count"].sum()
    total_train=train["Count"].sum()

    P_boys=boy_train/total_train
    P_girls=girl_train/total_train
    return P_boys,P_girls

def Likelihood(train,feature_column):
    likelihood={}
    gender=train.groupby("Gender")["Count"].sum()
    for feat in feature_column:
        counts=train.groupby(["Gender",feat],as_index=False)["Count"].sum()
        total=counts["Gender"].map(gender)
        d=train[feat].nunique()
        counts[f"P_{feat}_given"]=(counts["Count"]+1)/(total+d)
        likelihood[feat]=counts
    return likelihood

def build_lookup(likelihood,feature_column):
    lookup={}
    for feat in feature_column:
        df = likelihood[feat]
        lookup[feat] = df.set_index(["Gender", feat])[f"P_{feat}_given"].to_dict()
    return lookup

def predictGender(test,lookup,P_boys,P_girls,feature_column):
    logPriorBoys=np.log(P_boys)
    logPriorGirls=np.log(P_girls)
    N=len(test)
    F=len(feature_column)
    log_probM=np.zeros((N,F))
    log_probF=np.zeros((N,F))
    for i,feat in enumerate(feature_column):
        dictM={k[1]:np.log(v) for k,v in lookup[feat].items() if k[0]=="M"}
        dictF={k[1]:np.log(v) for k,v in lookup[feat].items() if k[0]=="F"}
        log_probM[:,i]=test[feat].map(dictM).fillna(np.log(1e-6)).values
        log_probF[:,i]=test[feat].map(dictF).fillna(np.log(1e-6)).values
    total_logM= logPriorBoys+np.sum(log_probM,axis=1)
    total_logF= logPriorGirls+np.sum(log_probF,axis=1)
    predictions=np.where(total_logM>total_logF,"M","F")
    test=test.copy()
    test["Predicted"]=predictions
    return test
def singleName(name,lookup,p_boys,p_girls):
    name = name.strip().lower()
    row = {"first_name":name[0],
           "last_two_letters":name[-2:],
           "length": len(name),
           "Vowels_Count": sum(1 for ch in name if ch in "aeiou")}
    logM = np.log(P_boys)
    logF = np.log(P_girls)
    for feat in feature_column:
            val=row[feat]
            p_m=lookup[feat].get(("M",val),1e-6)
            p_f=lookup[feat].get(("F",val),1e-6)
            logM+=np.log(p_m)
            logF+=np.log(p_f)
    return f"You are a good boy {name}" if logM>logF else f"You are a good girl {name}"

def evaluate(prediction):
    correct=(prediction["Gender"]==prediction["Predicted"]).sum()
    total=len(prediction)
    accuracy=correct/total
    return accuracy


name_gender=dataIngestion(folder)
features=VectorisedFeature(name_gender)
train,test=split(features)
P_boys,P_girls=Prior(train)
feature_column=["first_name","last_two_letters","length","Vowels_Count"]
likelihood=Likelihood(train,feature_column)
lookup = build_lookup(likelihood, feature_column)

predictions = predictGender(test,lookup,P_boys,P_girls,feature_column)
accuracy=evaluate(predictions)
result=singleName(input("The name you want to check "),lookup,P_boys,P_girls)

print(f"Accuracy = {accuracy*100}")
print(result)

# print(likelihood["last_name"])
# print(likelihood["length"])
# print(likelihood["Vowels_Count"])
# print (P_boys,P_girls,P_boys+P_girls)
# print(features.head(20))



Accuracy = 75.54573916026933
You are a good boy satyam
